In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Run from project root regardless of where the kernel was started.
if not (Path.cwd() / "data" / "aeroscapes").exists():
    os.chdir("..")

from src.config.aeroscapes_constants import IMAGENET_MEAN, IMAGENET_STD
from src.datasets.aeroscapes import AeroScapesDataset, build_train_transforms

ds = AeroScapesDataset(
    data_dir="data/aeroscapes",
    split="train",
    transforms=build_train_transforms(crop_size=512),
)

image, label_dict = ds[0]
mask = label_dict["mask"]

print("--- IMAGE ---")
print(f"dtype: {image.dtype}")  # torch.float32
print(f"shape: {image.shape}")  # [3, 512, 512]
print(f"min: {image.min():.4f}")  # ~ -2.x after ImageNet normalization
print(f"max: {image.max():.4f}")  # ~  2.x

print("\n--- MASK ---")
print(f"shape: {mask.shape}")  # [512, 512]
print(f"dtype: {mask.dtype}")  # torch.int64
print(f"unique classes: {mask.unique().tolist()}")

# De-normalize for display: x*std + mean -> [0, 1] range.
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
img_display = (image * std + mean).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_display)
axes[0].set_title("Image (de-normalized for display)")
axes[0].axis("off")
axes[1].imshow(mask.numpy(), cmap="nipy_spectral")
axes[1].set_title("Mask")
axes[1].axis("off")
plt.tight_layout()
plt.show()